# CozyDay Multi-Model AI System

## P1-A7 — Part 1

Team 3 | Stage 1: Lin

This notebook evaluates 15 Hugging Face models using 25 task-specific prompts, benchmarks inference performance, and simulates local hosting and API infrastructure costs.


## Step 1.1 — Model Size Categories

We reuse the same 15 Hugging Face models selected in A6.

| Category | Parameter Range | Models |
|---|---|---|
| Tiny | < 0.5B | 3 |
| Small | 0.5B?1.5B | 3 |
| Medium | > 1.5B?3B | 3 |
| Large | > 3B?7B | 3 |
| Very Large | > 7B | 3 |

Each category contains exactly three models.


In [1]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("results") if Path("results/research_matrix.csv").exists() else Path("llm-test/results")

model_matrix = pd.read_csv(RESULTS_DIR / "research_matrix.csv")

print("Total models:", len(model_matrix))
print("Columns:", model_matrix.columns.tolist())

display(model_matrix)

assert len(model_matrix) == 15, "Expected exactly 15 models"


Total models: 15
Columns: ['Model No.', 'Category', 'Model Name', 'Hugging Face URL', 'Parameters', 'License / Access', 'Hardware Requirements', 'Loading Instructions', 'Test Owner', 'Status']


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
0,1,Tiny (<0.5B),HuggingFaceTB/SmolLM2-135M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM2-1...,135M,Apache-2.0; public,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Lin,Tested (256-token baseline)
1,2,Small (0.5B-1.5B),Qwen/Qwen2.5-0.5B-Instruct,https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct,0.5B,Apache-2.0; public,CPU possible; RAM requirement to be benchmarked,AutoTokenizer + AutoModelForCausalLM,Lin,Tested (256-token baseline)
2,3,Medium (>1.5B-3B),Qwen/Qwen3-1.7B,https://huggingface.co/Qwen/Qwen3-1.7B,1.7B,Apache-2.0; public,Several GB RAM required; CPU may be slow,AutoTokenizer + AutoModelForCausalLM; check mo...,Lin,Tested (256-token baseline)
3,4,Large (>3B-7B),microsoft/Phi-3.5-mini-instruct,https://huggingface.co/microsoft/Phi-3.5-mini-...,3.8B,MIT; public,Approx. 8-14 GB FP16 weights; additional RAM n...,AutoTokenizer + AutoModelForCausalLM; review m...,Lin,"Tested (official FP16, Transformers 4.57.6, Co..."
4,5,Very Large (>7B),Qwen/Qwen3-8B,https://huggingface.co/Qwen/Qwen3-8B,8.2B,Apache-2.0; public,FP16 weights ~16.4 GB; additional RAM needed; ...,AutoTokenizer + AutoModelForCausalLM; follow m...,Lin,"Tested (official FP16, Transformers 4.57.6, Co..."
5,6,Tiny (<0.5B),HuggingFaceTB/SmolLM-135M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM-13...,135M,Apache-2.0; public,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Rishabh,"Tested (zero/one/few/many-shot, Transformers C..."
6,7,Small (0.5B-1.5B),Qwen/Qwen3-0.6B,https://huggingface.co/Qwen/Qwen3-0.6B,0.6B,Apache-2.0; public,CPU possible; RAM requirement to be benchmarked,AutoTokenizer + AutoModelForCausalLM,Rishabh,"Tested (zero/one/few/many-shot, Transformers C..."
7,8,Medium (>1.5B-3B),google/gemma-2-2b-it,https://huggingface.co/google/gemma-2-2b-it,~2B,Gemma license; acceptance required,Several GB RAM required; CPU may be slow,AutoTokenizer + AutoModelForCausalLM; check mo...,Rishabh,"Tested (zero/one/few/many-shot, Transformers C..."
8,9,Large (>3B-7B),Qwen/Qwen3-4B,https://huggingface.co/Qwen/Qwen3-4B,4B,Apache-2.0; public,Approx. 8-14 GB FP16 weights; additional RAM n...,AutoTokenizer + AutoModelForCausalLM; review m...,Rishabh,"Tested (zero/one/few/many-shot, Colab T4 FP16;..."
9,10,Very Large (>7B),mistralai/Mistral-Nemo-Instruct-2407,https://huggingface.co/mistralai/Mistral-Nemo-...,12B,Apache-2.0; public,FP16 weights ~24.0 GB; additional RAM needed; ...,AutoTokenizer + AutoModelForCausalLM; follow m...,Rishabh,"Tested (zero/one/few/many-shot, Colab T4, 4-bi..."


## Step 1.2 — Define Prompt Types

Five task-specific prompt categories are designed for CozyDay, an AI-powered daily planning assistant.

| Prompt Type | Description |
|---|---|
| Task Classification | Classify user tasks into meaningful categories. |
| Priority Ranking | Rank tasks based on urgency, deadlines, and importance. |
| Schedule Generation | Create realistic daily schedules based on time constraints. |
| Plan Summarization | Summarize daily activities and important commitments. |
| Structured Extraction | Extract structured task details from natural-language inputs. |

Each category will contain five evaluation prompts in Step 1.3.


## Step 1.3 — Create Evaluation Prompts

Five evaluation prompts are created for each of the five CozyDay task categories, giving 25 prompts total.


In [2]:

import pandas as pd

evaluation_prompts = [
    # Task Classification
    ("C1", "Task Classification", "Classify these tasks as Work, Study, Personal, or Health: finish project report, review lecture notes, buy groceries, and go for a run. Return one category per task."),
    ("C2", "Task Classification", "Categorize my tasks: email my supervisor, prepare for an exam, clean my room, and attend a dental appointment."),
    ("C3", "Task Classification", "Classify these activities: submit homework, schedule a team meeting, call my family, and attend a yoga class."),
    ("C4", "Task Classification", "Organize these tasks by category: write a research summary, cook dinner, review class slides, and visit the gym."),
    ("C5", "Task Classification", "Classify these tasks as Work, Study, Personal, or Health: update a client spreadsheet, study Python, pay rent, and attend a medical checkup."),

    # Priority Ranking
    ("P1", "Priority Ranking", "I have a report due tomorrow, groceries to buy, and a room to clean. Rank these tasks by priority and explain briefly."),
    ("P2", "Priority Ranking", "I have an exam tomorrow, a meeting next week, and laundry to finish. What should I prioritize today?"),
    ("P3", "Priority Ranking", "My presentation is due tonight, my emails can wait until Friday, and I need to exercise. Rank these tasks."),
    ("P4", "Priority Ranking", "I must submit an assignment by 5 PM, call a friend sometime this week, and prepare dinner tonight. Rank these tasks by urgency."),
    ("P5", "Priority Ranking", "I need to finish a group project due tomorrow, organize my desk, and prepare for a quiz next Monday. Give me a priority order."),

    # Schedule Generation
    ("S1", "Schedule Generation", "Create a schedule from 9 AM to 5 PM for studying 3 hours, exercising 1 hour, and grocery shopping 1 hour. Include breaks."),
    ("S2", "Schedule Generation", "Plan my day between 8 AM and 6 PM. I have class from 10 AM to noon, need 2 hours for homework, and want 1 hour for lunch."),
    ("S3", "Schedule Generation", "I am free from 1 PM to 9 PM. Schedule 3 hours of project work, 1 hour of exercise, and 1 hour of cooking."),
    ("S4", "Schedule Generation", "Build a realistic schedule from 9 AM to 7 PM with a meeting from 11 AM to noon, 3 hours of studying, and 2 hours of errands."),
    ("S5", "Schedule Generation", "Organize my afternoon from noon to 6 PM. I need 2 hours for research, 1 hour for groceries, and 1 hour for cleaning."),

    # Plan Summarization
    ("M1", "Plan Summarization", "Summarize my day: 9 AM lecture, 11 AM library study, 2 PM grocery shopping, and 6 PM dinner with friends."),
    ("M2", "Plan Summarization", "Give me a concise summary of tomorrow: morning workout, afternoon team meeting, evening homework."),
    ("M3", "Plan Summarization", "Summarize this schedule and highlight key commitments: 8 AM breakfast, 10 AM exam, 1 PM lunch, 3 PM office hours."),
    ("M4", "Plan Summarization", "Summarize my tasks in two sentences: finish readings, send emails, clean the apartment, and prepare meals."),
    ("M5", "Plan Summarization", "Provide a short daily overview: 9 AM project work, noon lunch, 2 PM class, 5 PM gym, and 7 PM dinner."),

    # Structured Extraction
    ("E1", "Structured Extraction", "Extract tasks and times as JSON from: I need to attend class at 9 AM, buy groceries at 4 PM, and study at 7 PM."),
    ("E2", "Structured Extraction", "Convert this into JSON with task and deadline fields: Finish the report by Friday, email the professor by Wednesday, and submit homework tomorrow."),
    ("E3", "Structured Extraction", "Extract tasks, durations, and categories into JSON: study Python for 2 hours, exercise for 45 minutes, and clean the kitchen for 30 minutes."),
    ("E4", "Structured Extraction", "Create a JSON task list from: Meeting at 10 AM, lunch at noon, project work at 2 PM, and dinner at 6 PM."),
    ("E5", "Structured Extraction", "Extract structured task data as JSON: submit the assignment by 11:59 PM, call my parents at 7 PM, and exercise for 30 minutes.")
]

prompts_df = pd.DataFrame(
    evaluation_prompts,
    columns=["Prompt ID", "Prompt Type", "Prompt"]
)

display(prompts_df)

assert len(prompts_df) == 25
assert prompts_df["Prompt Type"].nunique() == 5
assert all(prompts_df["Prompt Type"].value_counts() == 5)

print("Total prompts:", len(prompts_df))
print("Prompt types:", prompts_df["Prompt Type"].nunique())
print(prompts_df["Prompt Type"].value_counts())


,Prompt ID,Prompt Type,Prompt
0,C1,Task Classification,"Classify these tasks as Work, Study, Personal,..."
1,C2,Task Classification,"Categorize my tasks: email my supervisor, prep..."
2,C3,Task Classification,"Classify these activities: submit homework, sc..."
3,C4,Task Classification,Organize these tasks by category: write a rese...
4,C5,Task Classification,"Classify these tasks as Work, Study, Personal,..."
5,P1,Priority Ranking,"I have a report due tomorrow, groceries to buy..."
6,P2,Priority Ranking,"I have an exam tomorrow, a meeting next week, ..."
7,P3,Priority Ranking,"My presentation is due tonight, my emails can ..."
8,P4,Priority Ranking,"I must submit an assignment by 5 PM, call a fr..."
9,P5,Priority Ranking,"I need to finish a group project due tomorrow,..."


Total prompts: 25
Prompt types: 5
Prompt Type
Task Classification      5
Priority Ranking         5
Schedule Generation      5
Plan Summarization       5
Structured Extraction    5
Name: count, dtype: int64


## Step 1.4 — Define Evaluation Metrics

Each model output is evaluated using four equally weighted criteria.

| Metric | Weight | Description |
|---|---|---|
| Instruction Adherence | 25% | Follows all user instructions and constraints. |
| Accuracy | 25% | Correctly interprets tasks, deadlines, and scheduling constraints. |
| Clarity | 25% | Produces understandable and concise responses. |
| Structure | 25% | Uses the expected output format. |

### Scoring Convention

Each criterion receives an integer score from 1 to 10.

- 1?2: Very poor
- 3?4: Poor
- 5?6: Acceptable
- 7?8: Good
- 9?10: Excellent

Overall Score = (Instruction Adherence + Accuracy + Clarity + Structure) / 4

The final score is rounded to the nearest integer using round-half-up.

Scores must be assigned after reviewing actual model outputs.
Failed inference runs receive a missing score (NaN), not an invented score.


In [3]:
import math
import pandas as pd

EVALUATION_METRICS = [
    "Instruction Adherence",
    "Accuracy",
    "Clarity",
    "Structure"
]

def calculate_score(instruction, accuracy, clarity, structure):
    scores = [instruction, accuracy, clarity, structure]

    if not all(isinstance(x, int) and 1 <= x <= 10 for x in scores):
        raise ValueError("Each metric must be an integer between 1 and 10.")

    average = sum(scores) / len(scores)

    return math.floor(average + 0.5)

# Verify scoring function
assert calculate_score(8, 9, 7, 8) == 8
assert calculate_score(9, 9, 10, 10) == 10

print("Evaluation metrics:", EVALUATION_METRICS)
print("Example overall score:", calculate_score(8, 9, 7, 8))
print("Scoring system ready!")


Evaluation metrics: ['Instruction Adherence', 'Accuracy', 'Clarity', 'Structure']
Example overall score: 8
Scoring system ready!


## Step 1.5 — Benchmark Experiments

Stage 1: benchmark Models 1-5 on the shared 25 prompts.


In [ ]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Run the earlier Step 1.1 and Step 1.3 cells first.
assert len(prompts_df) == 25
assert len(model_matrix) == 15

RESULTS_DIR = Path("results") if Path("results").is_dir() else Path("llm-test/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_IDS = {
    1: "HuggingFaceTB/SmolLM2-135M-Instruct",
    2: "Qwen/Qwen2.5-0.5B-Instruct",
    3: "Qwen/Qwen3-1.7B",
    4: "microsoft/Phi-3.5-mini-instruct",
    5: "Qwen/Qwen3-8B",
}

MAX_NEW_TOKENS = 256
SYSTEM_INSTRUCTION = (
    "You are CozyDay, a helpful daily planning assistant. "
    "Use only the information given by the user. "
    "Follow the requested format. Be accurate and concise."
)

def run_a7_benchmark(model_no):
    if model_no not in MODEL_IDS:
        raise ValueError("Only Stage 1 models (1-5) are supported.")

    model_id = MODEL_IDS[model_no]
    output_path = RESULTS_DIR / f"a7_model_{model_no}_benchmark.csv"

    # Resume completed prompts without overwriting measurements.
    if output_path.exists():
        results = pd.read_csv(output_path)
        if "Prompt ID" not in results.columns:
            raise ValueError("Existing result file lacks Prompt ID.")
        if results["Prompt ID"].duplicated().any():
            raise ValueError("Duplicate Prompt IDs in saved results.")
        completed = set(results["Prompt ID"].dropna())
        print(f"Resuming {len(completed)} previously completed prompts.")
    else:
        results = pd.DataFrame()
        completed = set()

    remaining = prompts_df[
        ~prompts_df["Prompt ID"].isin(completed)
    ]

    if remaining.empty:
        print("All 25 prompts are already complete.")
        return results

    print(f"Loading Model {model_no}: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, local_files_only=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, local_files_only=True
    )
    model.eval()
    device = next(model.parameters()).device

    try:
        for _, item in remaining.iterrows():
            prompt_id = item["Prompt ID"]
            print(f"Running {prompt_id}...", flush=True)

            messages = [
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": item["Prompt"]},
            ]

            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
                **(
                    {"enable_thinking": False}
                    if "Qwen3" in model_id else {}
                ),
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            input_length = inputs["input_ids"].shape[-1]

            start = time.perf_counter()
            with torch.inference_mode():
                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            latency = time.perf_counter() - start

            generated = output[0][input_length:]
            token_count = len(generated)
            response = tokenizer.decode(
                generated, skip_special_tokens=True
            )

            row = {
                "Model": model_id,
                "Model Category": model_matrix.loc[
                    model_matrix["Model No."] == model_no,
                    "Category"
                ].iloc[0],
                "Tokens/sec": round(
                    token_count / latency, 3
                ) if latency > 0 else 0,
                "Latency": round(latency, 3),
                "Prompt Type": item["Prompt Type"],
                "Prompt": item["Prompt"],
                "Self-Evaluation Score (1-10)": None,
                "Prompt ID": prompt_id,
                "Response": response,
                "Generated Tokens": token_count,
                "Reached Token Limit": token_count >= MAX_NEW_TOKENS,
                "Backend": "Transformers",
                "Device": str(device),
                "Precision": str(next(model.parameters()).dtype),
                "Max New Tokens": MAX_NEW_TOKENS,
                "Instruction Adherence": None,
                "Accuracy": None,
                "Clarity": None,
                "Structure": None,
            }

            results = pd.concat(
                [results, pd.DataFrame([row])],
                ignore_index=True
            )
            results.to_csv(output_path, index=False)

            print(
                f"{prompt_id} complete | "
                f"{latency:.2f}s | "
                f"{row['Tokens/sec']:.2f} tokens/sec",
                flush=True
            )

        print(f"Saved {len(results)} results to {output_path}")
        return results

    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# Stage 1 — Model 3 Benchmark
# Models 1 and 2 have already been completed.

RUN_MODEL_3 = False

if RUN_MODEL_3:
    model_3_a7_results = run_a7_benchmark(3)
    display(model_3_a7_results.tail())
else:
    print("Benchmark runner ready. Set RUN_MODEL_3 = False to start.")


Runner defined; inference opt-in and disabled.


In [8]:

import math
from pathlib import Path
import pandas as pd
from IPython.display import display

# ==========================================
# A7 Stage 1 — Shared Cost Assumptions
# ==========================================

AVG_TOKENS_PER_REQUEST = 500
REQUESTS_PER_USER_PER_DAY = 1
DAYS_PER_MONTH = 30

# Cloud GPU reference: AWS g5.xlarge
# A10G 24GB, $1.006/hour (us-east-1, on-demand)
CLOUD_HOURLY_RATE = 1.006

# Illustrative utilization and throughput assumptions.
# These are NOT measured cloud GPU benchmarks.
UTILIZATION = 0.70
ESTIMATED_CLOUD_TOKENS_SEC = {
    1: 60,
    2: 45,
    3: 30,
    4: 20,
    5: 12
}

# Hypothetical paid API output-token pricing.
# USD per 1 million output tokens.
# NOT verified Hugging Face provider quotes.
ASSUMED_API_PER_MILLION = {
    1: 0.10,
    2: 0.20,
    3: 0.40,
    4: 0.75,
    5: 1.20
}

# Illustrative API gateway:
# $0.05/hour per gateway, 100 requests/sec capacity.
API_GATEWAY_HOURLY = 0.05
API_GATEWAY_RPS = 100

TRAFFIC = {
    "Prototype": 1000,
    "Early Startup": 10000,
    "Growing Product": 100000,
    "Large Platform": 1000000,
    "Mass Consumer App": 10000000,
    "Global Platform": 100000000
}

MODEL_NAMES = {
    1: "HuggingFaceTB/SmolLM2-135M-Instruct",
    2: "Qwen/Qwen2.5-0.5B-Instruct",
    3: "Qwen/Qwen3-1.7B",
    4: "microsoft/Phi-3.5-mini-instruct",
    5: "Qwen/Qwen3-8B"
}

local_rows = []
api_rows = []

for model_no, model_name in MODEL_NAMES.items():
    for category, users in TRAFFIC.items():

        daily_requests = users * REQUESTS_PER_USER_PER_DAY
        daily_tokens = daily_requests * AVG_TOKENS_PER_REQUEST

        # LOCAL HOSTING
        capacity = (
            ESTIMATED_CLOUD_TOKENS_SEC[model_no]
            * 86400
            * UTILIZATION
        )

        machines = max(1, math.ceil(daily_tokens / capacity))
        monthly_machine_cost = (
            machines * CLOUD_HOURLY_RATE * 24 * DAYS_PER_MONTH
        )

        local_rows.append({
            "Model Name": model_name,
            "Daily Traffic Category": category,
            "Daily Active Users": users,
            "Avg Tokens per User": AVG_TOKENS_PER_REQUEST,
            "Total Daily Token Load": daily_tokens,
            "Per Machine Cost (Cloud)": CLOUD_HOURLY_RATE,
            "Machines Needed": machines,
            "Total Machine Cost": round(monthly_machine_cost, 2),
            "Cloud Throughput Assumption (tokens/sec)":
                ESTIMATED_CLOUD_TOKENS_SEC[model_no],
            "Cost Period": "USD/month"
        })

        # HUGGING FACE API SCENARIO
        api_rate = ASSUMED_API_PER_MILLION[model_no]

        daily_api_cost = daily_tokens / 1_000_000 * api_rate
        monthly_api_cost = daily_api_cost * DAYS_PER_MONTH

        gateway_machines = max(
            1,
            math.ceil(
                daily_requests / (86400 * API_GATEWAY_RPS)
            )
        )

        gateway_monthly_cost = (
            gateway_machines
            * API_GATEWAY_HOURLY
            * 24
            * DAYS_PER_MONTH
        )

        api_rows.append({
            "Local LLM Model Size": model_name,
            "Comparable Paid Model Hosted on Hugging Face":
                model_name + " (hypothetical hosted equivalent)",
            "Cost of Paid API per Token": api_rate / 1_000_000,
            "Daily Traffic Category": category,
            "Total Token Load": daily_tokens,
            "Estimated API Cost": round(monthly_api_cost, 2),
            "Machines Required (API infrastructure)": gateway_machines,
            "Total Cost": round(
                monthly_api_cost + gateway_monthly_cost, 2
            ),
            "API Gateway Cost": round(gateway_monthly_cost, 2),
            "Pricing Status": "Illustrative assumption, not quoted",
            "Cost Period": "USD/month"
        })

local_cost_df = pd.DataFrame(local_rows)
api_cost_df = pd.DataFrame(api_rows)

# ==========================================
# Save Results
# ==========================================

if Path("llm-test/results").exists():
    output_dir = Path("llm-test/results")
else:
    output_dir = Path("results")

output_dir.mkdir(parents=True, exist_ok=True)

local_cost_df.to_csv(
    output_dir / "a7_stage1_local_cost.csv", index=False
)
api_cost_df.to_csv(
    output_dir / "a7_stage1_api_cost.csv", index=False
)

# ==========================================
# Verification
# ==========================================

assert len(local_cost_df) == 30
assert len(api_cost_df) == 30
assert local_cost_df["Total Daily Token Load"].gt(0).all()
assert api_cost_df["Total Token Load"].gt(0).all()

print("Local Hosting Cost Rows:", len(local_cost_df))
print("Hugging Face API Scenario Rows:", len(api_cost_df))
print("Total Cost Rows:", len(local_cost_df) + len(api_cost_df))
print("Average Tokens per Request:", AVG_TOKENS_PER_REQUEST)
print("Saved to:", output_dir)

display(local_cost_df.head(6))
display(api_cost_df.head(6))


Local Hosting Cost Rows: 30
Hugging Face API Scenario Rows: 30
Total Cost Rows: 60
Average Tokens per Request: 500
Saved to: results


,Model Name,Daily Traffic Category,Daily Active Users,Avg Tokens per User,Total Daily Token Load,Per Machine Cost (Cloud),Machines Needed,Total Machine Cost,Cloud Throughput Assumption (tokens/sec),Cost Period
0,HuggingFaceTB/SmolLM2-135M-Instruct,Prototype,1000,500,500000,1.006,1,724.32,60,USD/month
1,HuggingFaceTB/SmolLM2-135M-Instruct,Early Startup,10000,500,5000000,1.006,2,1448.64,60,USD/month
2,HuggingFaceTB/SmolLM2-135M-Instruct,Growing Product,100000,500,50000000,1.006,14,10140.48,60,USD/month
3,HuggingFaceTB/SmolLM2-135M-Instruct,Large Platform,1000000,500,500000000,1.006,138,99956.16,60,USD/month
4,HuggingFaceTB/SmolLM2-135M-Instruct,Mass Consumer App,10000000,500,5000000000,1.006,1378,998112.96,60,USD/month
5,HuggingFaceTB/SmolLM2-135M-Instruct,Global Platform,100000000,500,50000000000,1.006,13779,9980405.28,60,USD/month


,Local LLM Model Size,Comparable Paid Model Hosted on Hugging Face,Cost of Paid API per Token,Daily Traffic Category,Total Token Load,Estimated API Cost,Machines Required (API infrastructure),Total Cost,API Gateway Cost,Pricing Status,Cost Period
0,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct (hypotheti...,1.000000e-07,Prototype,500000,1.5,1,37.5,36.0,"Illustrative assumption, not quoted",USD/month
1,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct (hypotheti...,1.000000e-07,Early Startup,5000000,15.0,1,51.0,36.0,"Illustrative assumption, not quoted",USD/month
2,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct (hypotheti...,1.000000e-07,Growing Product,50000000,150.0,1,186.0,36.0,"Illustrative assumption, not quoted",USD/month
3,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct (hypotheti...,1.000000e-07,Large Platform,500000000,1500.0,1,1536.0,36.0,"Illustrative assumption, not quoted",USD/month
4,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct (hypotheti...,1.000000e-07,Mass Consumer App,5000000000,15000.0,2,15072.0,72.0,"Illustrative assumption, not quoted",USD/month
5,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct (hypotheti...,1.000000e-07,Global Platform,50000000000,150000.0,12,150432.0,432.0,"Illustrative assumption, not quoted",USD/month


In [9]:

import pandas as pd
from pathlib import Path
from IPython.display import display

# Find the existing A7 results directory.
candidates = [
    Path("results"),
    Path("llm-test/results")
]

RESULTS_DIR = next(
    (p for p in candidates if
     (p / "a7_model_1_scored.csv").exists()
     and (p / "a7_stage1_local_cost.csv").exists()),
    None
)

if RESULTS_DIR is None:
    raise FileNotFoundError("A7 results folder not found.")

print("Using results folder:", RESULTS_DIR)

# 1. Load the five scored benchmark files.
benchmark_frames = []

for model_no in range(1, 6):
    path = RESULTS_DIR / f"a7_model_{model_no}_scored.csv"
    df = pd.read_csv(path)

    assert len(df) == 25, f"Model {model_no}: expected 25 rows"
    assert df["Prompt ID"].nunique() == 25
    assert df["Self-Evaluation Score (1-10)"].notna().all()
    assert df["Self-Evaluation Score (1-10)"].between(1, 10).all()

    df.insert(0, "Model No.", model_no)
    benchmark_frames.append(df)

stage1_benchmark_df = pd.concat(
    benchmark_frames,
    ignore_index=True
)

# 2. Load cost tables.
stage1_local_cost_df = pd.read_csv(
    RESULTS_DIR / "a7_stage1_local_cost.csv"
)
stage1_api_cost_df = pd.read_csv(
    RESULTS_DIR / "a7_stage1_api_cost.csv"
)

# 3. Validate model and prompt coverage.
assert len(stage1_benchmark_df) == 125
assert len(stage1_local_cost_df) == 30
assert len(stage1_api_cost_df) == 30

assert stage1_benchmark_df[
    ["Model No.", "Prompt ID"]
].drop_duplicates().shape[0] == 125

expected_prompts = set(stage1_benchmark_df[
    stage1_benchmark_df["Model No."] == 1
]["Prompt ID"])

for model_no in range(1, 6):
    observed = set(stage1_benchmark_df[
        stage1_benchmark_df["Model No."] == model_no
    ]["Prompt ID"])
    assert observed == expected_prompts

# 4. Display summary.
stage1_summary = (
    stage1_benchmark_df
    .groupby(["Model No.", "Model", "Model Category"])
    .agg(
        Prompts=("Prompt ID", "count"),
        Average_Score=("Self-Evaluation Score (1-10)", "mean"),
        Average_Latency=("Latency", "mean"),
        Average_Tokens_Per_Sec=("Tokens/sec", "mean"),
        Token_Limit_Hits=("Reached Token Limit", "sum")
    )
    .reset_index()
    .round(2)
)

display(stage1_summary)

# 5. Save shared Stage 1 datasets.
stage1_benchmark_df.to_csv(
    RESULTS_DIR / "a7_stage1_all_benchmarks.csv",
    index=False
)

stage1_summary.to_csv(
    RESULTS_DIR / "a7_stage1_summary.csv",
    index=False
)

print("\n=== Stage 1 Verification ===")
print("Benchmark rows:", len(stage1_benchmark_df))
print("Local cost rows:", len(stage1_local_cost_df))
print("API cost rows:", len(stage1_api_cost_df))
print("Total:", (
    len(stage1_benchmark_df)
    + len(stage1_local_cost_df)
    + len(stage1_api_cost_df)
))
print("All five models verified!")


Using results folder: results


,Model No.,Model,Model Category,Prompts,Average_Score,Average_Latency,Average_Tokens_Per_Sec,Token_Limit_Hits
0,1,HuggingFaceTB/SmolLM2-135M-Instruct,Tiny (<0.5B),25,2.76,5.19,24.49,10
1,2,Qwen/Qwen2.5-0.5B-Instruct,Small (0.5B-1.5B),25,6.80,5.56,12.37,1
2,3,Qwen/Qwen3-1.7B,Medium (>1.5B-3B),25,8.00,17.34,4.34,2
3,4,microsoft/Phi-3.5-mini-instruct,Large (>3B-7B),25,9.40,6.05,24.21,3
4,5,Qwen/Qwen3-8B,Very Large (>7B),25,9.64,6.66,11.76,1



=== Stage 1 Verification ===
Benchmark rows: 125
Local cost rows: 30
API cost rows: 30
Total: 185
All five models verified!


## A7 Stage 1 corrections and Stage 2/3 reusable interface

Use `a7_shared.py` for future benchmarking and costs. **Do not run inference by default.** Run from either the `llm-test` directory or the repo root with `llm-test` on `sys.path`. Saved A7 Stage 1 benchmark/scored CSVs are unchanged. Model registry automatically contains all 15 A6 models. For GPU, specify FP16 or NF4 explicitly; any new environment must be benchmarked and documented separately. The built-in A6 matrix `Status` metadata is NOT A7 execution metadata. The original runner remains solely for reproducing historical Stage 1 local tests.

**Recorded A7 backends:** Models 1–3: CPU, `torch.bfloat16` per stored CSVs; Models 4–5: Colab T4 (14.56 GB). Model 4: native Phi-3.5 implementation (`trust_remote_code=False`), FP16, eager attention. Model 5: Qwen3 thinking disabled, 4-bit NF4 with double quantization, FP16 compute, `device_map={'':0}`. Colab reported Python 3.13.15, torch 2.11.0+cu130, Transformers 5.20.1, accelerate 1.15.0, bitsandbytes 0.50.2; local torch 2.14.1+cpu, Transformers 5.19.0. Original Colab GPU timing synchronized CUDA around `generate()`. Warmup, CPU threading and mixed precisions limit speed comparisons.

**Scoring calibration:** Follow the four equal-weight 1–10 criteria. Classification must cover every task; prioritization must respect hard deadlines; schedules must have correct total durations without time collisions; summaries must obey sentence count; JSON extraction must include every requested task and avoid invented details. Truncation, omissions and invalid JSON should reduce instruction/structure ratings. Current manually assigned Stage 1 scores are preserved pending adjudication of Model 2 P5, Model 3 M4, Model 4 S5 and Model 5 S1.


In [ ]:
import sys
from pathlib import Path
here = Path.cwd()
module_dir = here if (here/'a7_shared.py').exists() else here/'llm-test'
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
from a7_shared import registry, run_models, calculate_costs
assert len(registry(model_matrix)) == 15
# No inference is performed on Run All.
print('Stage 2/3 shared functions available; inference disabled by default.')
